# Epi Info AI frequency validation lab — V0.9

Validate the candidate `epi.frequency` Rust/WebAssembly kernel against the canonical foodborne data and independent SciPy formulas. Passing is evidence, not statistical approval; G5 is deferred to the consolidated review of all outputs.

In [ ]:
import csv, hashlib, io, math
from pyodide.http import pyfetch
from js import WebAssembly, Uint8Array
fixture_response = await pyfetch('../../validation-fixtures/foodborne-frequency-v0.9.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
data_response = await pyfetch('../../examples/foodborne-outbreak-investigation.csv')
data_response.raise_for_status(); data_bytes = await data_response.bytes()
assert hashlib.sha256(data_bytes).hexdigest() == fixture['dataset']['sha256']
records = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
assert len(records) == fixture['dataset']['rows'] == 96
wasm_response = await pyfetch('../../epi2x2.wasm')
instance = await WebAssembly.instantiate(Uint8Array.new(await wasm_response.buffer()), {})
rust = instance.instance.exports


In [ ]:
from collections import Counter
counts = Counter(record[fixture['request']['sourceHeader']] for record in records)
expected_counts = {item['value']: item['frequency'] for item in fixture['expected']['categories']}
assert dict(sorted(counts.items())) == expected_counts
counts

In [ ]:
from scipy.stats import beta
n = fixture['expected']['includedRecords']; alpha = 0.05
for item in fixture['expected']['categories']:
    x = item['frequency']
    lower = 0.0 if x == 0 else float(beta.ppf(alpha / 2, x, n - x + 1))
    upper = 1.0 if x == n else float(beta.ppf(1 - alpha / 2, x + 1, n - x))
    assert math.isclose(x / n, item['percent'], abs_tol=1e-14, rel_tol=0)
    assert math.isclose(lower, item['lower'], abs_tol=1e-12, rel_tol=0)
    assert math.isclose(upper, item['upper'], abs_tol=1e-12, rel_tol=0)
print('PASS: foodborne counts and independent Clopper-Pearson anchors')


In [ ]:
cumulative = 0
for item in fixture['expected']['categories']:
    x = item['frequency']; cumulative += x
    assert math.isclose(float(rust.frequency_proportion(x, n)), item['percent'], abs_tol=1e-14, rel_tol=0)
    assert math.isclose(float(rust.frequency_proportion(cumulative, n)), item['cumulativePercent'], abs_tol=1e-14, rel_tol=0)
    assert math.isclose(float(rust.frequency_ci_lower(x, n)), item['lower'], abs_tol=1e-12, rel_tol=0)
    assert math.isclose(float(rust.frequency_ci_upper(x, n)), item['upper'], abs_tol=1e-12, rel_tol=0)
assert float(rust.frequency_ci_lower(300, 300)) == 1.0
assert float(rust.frequency_ci_upper(300, 300)) == 1.0
assert math.isnan(float(rust.frequency_ci_lower(11, 10)))
print('PASS: deployed Rust/WASM matches V0.9 anchors and audited boundary behavior')
